In [8]:
%%capture
!pip install torchmetrics

In [9]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from tqdm.notebook import tqdm
from collections import defaultdict

from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


from IPython.display import clear_output
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()

from torchmetrics import F1Score



import nltk
from nltk.probability import FreqDist
from nltk.tokenize import word_tokenize

In [10]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [11]:

df = pd.read_csv('trip-advisor-hotel-reviews.zip')

In [12]:
df.head()

,Review,Rating
0,nice hotel expensive parking got good deal sta...,4
1,ok nothing special charge diamond member hilto...,2
2,nice rooms not 4* experience hotel monaco seat...,3
3,"unique, great stay, wonderful time hotel monac...",5
4,"great stay great stay, went seahawk game aweso...",5


In [13]:
X = df['Review']
y = df['Rating']

In [14]:
X.head()

,Review
0,nice hotel expensive parking got good deal sta...
1,ok nothing special charge diamond member hilto...
2,nice rooms not 4* experience hotel monaco seat...
3,"unique, great stay, wonderful time hotel monac..."
4,"great stay great stay, went seahawk game aweso..."


In [15]:
y.head()

,Rating
0,4
1,2
2,3
3,5
4,5


In [26]:
%%capture
!pip install pytorch-lightning

In [36]:
df_train, df_temp = train_test_split(df, test_size=0.2, random_state=42)
df_val, df_test = train_test_split(df_temp, test_size=0.25, random_state=42)

max_words = 10000

tokens = []

for text in tqdm(df_train['Review']):
  tokens.extend(word_tokenize(text))

tokens_filtered = [word for word in tokens if word.isalnum()]

dist = FreqDist(tokens_filtered)
tokens_filtered_top = [pair[0] for pair in dist.most_common(max_words-1)]
vocabulary = {v: k for k, v in dict(enumerate(tokens_filtered_top, 1)).items()}

def text_to_sequence(text, maxlen):
    result = []
    tokens = word_tokenize(text.lower())
    tokens_filtered = [word for word in tokens if word.isalnum()]
    for word in tokens_filtered:
        if word in vocabulary:
            result.append(vocabulary[word])
    padding = [0]*(maxlen-len(result))
    return padding + result[-maxlen:]

  0%|          | 0/16392 [00:00<?, ?it/s]

In [28]:
import pytorch_lightning as pl
from torch.utils.data import DataLoader, Dataset
import torch
import numpy as np

class TextDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.from_numpy(x).long()
        self.y = torch.from_numpy(y).long()

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


class HotelReviewDataModule(pl.LightningDataModule):
    def __init__(self, df_train, df_val, df_test, max_len=100, batch_size=256):
        super().__init__()
        self.df_train = df_train
        self.df_val = df_val
        self.df_test = df_test
        self.max_len = max_len
        self.batch_size = batch_size

    def setup(self, stage=None):
        if stage == "fit" or stage is None:
            x_train = np.array([text_to_sequence(text, self.max_len)
                               for text in self.df_train["Review"]], dtype=np.int32)
            y_train = np.array(self.df_train["Rating"]) - 1

            x_val = np.array([text_to_sequence(text, self.max_len)
                             for text in self.df_val["Review"]], dtype=np.int32)
            y_val = np.array(self.df_val["Rating"]) - 1

            self.train_dataset = TextDataset(x_train, y_train)
            self.val_dataset = TextDataset(x_val, y_val)

        if stage == "test" or stage is None:
            x_test = np.array([text_to_sequence(text, self.max_len)
                              for text in self.df_test["Review"]], dtype=np.int32)
            y_test = np.array(self.df_test["Rating"]) - 1

            self.test_dataset = TextDataset(x_test, y_test)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size)

In [37]:
import pytorch_lightning as pl
import torch
import torch.nn as nn
from torchmetrics import Accuracy, F1Score

class ConvTextClassifierLightning(pl.LightningModule):
    def __init__(self, vocab_size=2000, embedding_dim=32, out_channel=128, num_classes=5, learning_rate=0.001):
        super().__init__()
        self.save_hyperparameters()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.conv = nn.Conv1d(embedding_dim, out_channel, kernel_size=3)
        self.classifier = nn.Sequential(
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(out_channel, num_classes),
        )

        self.criterion = nn.CrossEntropyLoss()

        self.train_acc = Accuracy(task="multiclass", num_classes=num_classes)
        self.val_acc = Accuracy(task="multiclass", num_classes=num_classes)
        self.test_acc = Accuracy(task="multiclass", num_classes=num_classes)

        self.train_f1 = F1Score(task="multiclass", num_classes=num_classes)
        self.val_f1 = F1Score(task="multiclass", num_classes=num_classes)
        self.test_f1 = F1Score(task="multiclass", num_classes=num_classes)

    def forward(self, x):
        x = self.embedding(x).transpose(1, 2)
        x = self.conv(x)
        x = x.amax(dim=2)
        return self.classifier(x)

    def _shared_step(self, batch, prefix: str):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = logits.argmax(dim=1)

        if prefix == "train":
            acc, f1 = self.train_acc, self.train_f1
        elif prefix == "val":
            acc, f1 = self.val_acc, self.val_f1
        else:
            acc, f1 = self.test_acc, self.test_f1

        acc(preds, y)
        f1(preds, y)

        self.log(f"{prefix}_loss", loss, prog_bar=True)
        self.log(f"{prefix}_acc", acc, prog_bar=True)
        self.log(f"{prefix}_f1", f1, prog_bar=True)

        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self._shared_step(batch, "val")

    def test_step(self, batch, batch_idx):
        return self._shared_step(batch, "test")

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=2
        )
        return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "monitor": "val_loss"}}

In [38]:
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

data_module = HotelReviewDataModule(df_train, df_val, df_test, max_len=100, batch_size=256)

model = ConvTextClassifierLightning(
    vocab_size=10000,
    embedding_dim=64,
    out_channel=128,
    num_classes=5,
    learning_rate=0.001
)

early_stop_callback = EarlyStopping(
    monitor='val_loss',
    patience=5,
    mode='min'
)

trainer = pl.Trainer(
    max_epochs=20,
    callbacks=[early_stop_callback],
    accelerator='gpu',
    log_every_n_steps=10
)

trainer.fit(model, data_module)
trainer.test(model, data_module)

INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embedding  │ Embedding          │  640 K │ train │     0 │
│ 1 │ conv       │ Conv1d             │ 24.7 K │ train │     0 │
│ 2 │ classifier │ Sequential         │    645 │ train │     0 │
│ 3 │ criterion  │ CrossEntropyLoss   │      0 │ train │     0 │
│ 4 │ train_acc  │ MulticlassAccuracy │      0 │ train │     0 │
│ 5 │ val_acc    │ MulticlassAccuracy │      0 │ train │     0 │
│ 6 │ test_acc   │ MulticlassAccuracy │      0 │ train │     0 │
│ 7 │ train_f1   │ MulticlassF1Score  │      0 │ train │     0 │
│ 8 │ val_f1     │ MulticlassF1Score  │      0 │ train │     0 │
│ 9 │ test_f1    │ MulticlassF1Score  │      0 │ train │     0 │
└───┴────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 665 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 665 K                                                                                                
Total estimated model params size (MB): 2                                                                          
Modules in train mode: 13                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=20` reached.


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.6029268503189087     │
│          test_f1          │    0.6029268503189087     │
│         test_loss         │    0.9299784302711487     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.9299784302711487,
  'test_acc': 0.6029268503189087,
  'test_f1': 0.6029268503189087}]

In [39]:
import torch
import torch.nn as nn

class LSTMTextClassifierLightning(ConvTextClassifierLightning):
    def __init__(
        self,
        vocab_size=5000,
        embedding_dim=64,
        hidden_size=128,
        num_classes=5,
        learning_rate=0.001,
        num_layers=2,
    ):
        super().__init__(num_classes=num_classes, learning_rate=learning_rate)
        self.save_hyperparameters()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3 if num_layers > 1 else 0.0,
            bidirectional=True,
        )

        self.pool = nn.AdaptiveMaxPool1d(1)

        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size * 2),
            nn.Dropout(0.5),
            nn.Linear(hidden_size * 2, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_size, num_classes),
        )

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        out = out.transpose(1, 2)
        out = self.pool(out).squeeze(-1)
        return self.head(out)

In [40]:
lstm_model = LSTMTextClassifierLightning(vocab_size=10000, num_classes=5)


early_stop_callback = EarlyStopping(
    monitor='val_loss',
    patience=5,
    mode='min'
)

trainer = pl.Trainer(
    max_epochs=20,
    callbacks=[early_stop_callback],
    accelerator='gpu',
    log_every_n_steps=10
)


trainer.fit(lstm_model, data_module)
results_lstm = trainer.test(lstm_model, data_module)

print("LSTM:", results_lstm)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name       ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ embedding  │ Embedding          │  640 K │ train │     0 │
│ 1  │ conv       │ Conv1d             │ 12.4 K │ train │     0 │
│ 2  │ classifier │ Sequential         │    645 │ train │     0 │
│ 3  │ criterion  │ CrossEntropyLoss   │      0 │ train │     0 │
│ 4  │ train_acc  │ MulticlassAccuracy │      0 │ train │     0 │
│ 5  │ val_acc    │ MulticlassAccuracy │      0 │ train │     0 │
│ 6  │ test_acc   │ MulticlassAccuracy │      0 │ train │     0 │
│ 7  │ train_f1   │ MulticlassF1Score  │      0 │ train │     0 │
│ 8  │ val_f1     │ MulticlassF1Score  │      0 │ train │     0 │
│ 9  │ test_f1    │ MulticlassF1Score  │      0 │ train │     0 │
│ 10 │ lstm       │ LSTM               │  593 K │ train │     0 │
│ 11 │ pool       │ AdaptiveMaxPool1d  │      0 │ train │     0 │
│ 12 │ head       │ Sequential         │ 34.1 K │ train │     0 │
└────┴────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 1.3 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.3 M                                                                                                
Total estimated model params size (MB): 5                                                                          
Modules in train mode: 22                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │     0.58829265832901      │
│          test_f1          │     0.58829265832901      │
│         test_loss         │    1.0815225839614868     │
└───────────────────────────┴───────────────────────────┘

LSTM: [{'test_loss': 1.0815225839614868, 'test_acc': 0.58829265832901, 'test_f1': 0.58829265832901}]


 Сначала Embedding превращает каждый индекс в вектор, затем эти эмбеддинги проходят через dropout. Дальше параллельно применяются три Conv1d со свёртками разных размеров окна (3, 4 и 5), после каждой свёртки идёт BatchNorm и активация GELU. Из каждого набора признаков берётся глобальный максимум по всей длине текста (AdaptiveMaxPool1d(1)), поэтому для каждого окна получается один вектор признаков. Эти три вектора конкатенируются в один общий и подаются в “голову” классификатора: dropout → линейный слой до 256 → LayerNorm → GELU → dropout → финальный линейный слой, который выдаёт 5 логитов

In [42]:
import torch
import torch.nn as nn
from torchmetrics import Accuracy, F1Score

class CNNV2(ConvTextClassifierLightning):
    def __init__(self, vocab_size=10000, embedding_dim=128, out_channel=256, num_classes=5, learning_rate=0.001):
        super().__init__(num_classes=num_classes, learning_rate=learning_rate)
        self.save_hyperparameters()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        self.convs = nn.ModuleList([
            nn.Conv1d(embedding_dim, out_channel, kernel_size=3, padding=1),
            nn.Conv1d(embedding_dim, out_channel, kernel_size=4, padding=2),
            nn.Conv1d(embedding_dim, out_channel, kernel_size=5, padding=2),
        ])
        self.bns = nn.ModuleList([nn.BatchNorm1d(out_channel) for _ in range(len(self.convs))])

        self.act = nn.GELU()
        self.drop_emb = nn.Dropout(0.2)

        self.pool = nn.AdaptiveMaxPool1d(1)

        self.head = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(out_channel * len(self.convs), 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.embedding(x)
        x = self.drop_emb(x)
        x = x.transpose(1, 2)

        pooled = []
        for conv, bn in zip(self.convs, self.bns):
            h = self.act(bn(conv(x)))
            h = self.pool(h).squeeze(-1)
            pooled.append(h)

        x = torch.cat(pooled, dim=1)
        return self.head(x)

In [43]:
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

data_module = HotelReviewDataModule(df_train, df_val, df_test, max_len=100, batch_size=256)

model = CNNV2(
    vocab_size=10000,
    embedding_dim=64,
    out_channel=128,
    num_classes=5,
    learning_rate=0.001
)

early_stop_callback = EarlyStopping(
    monitor='val_loss',
    patience=5,
    mode='min'
)

trainer = pl.Trainer(
    max_epochs=20,
    callbacks=[early_stop_callback],
    accelerator='gpu',
    log_every_n_steps=10
)

trainer.fit(model, data_module)
trainer.test(model, data_module)

INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name       ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ embedding  │ Embedding          │  640 K │ train │     0 │
│ 1  │ conv       │ Conv1d             │ 12.4 K │ train │     0 │
│ 2  │ classifier │ Sequential         │    645 │ train │     0 │
│ 3  │ criterion  │ CrossEntropyLoss   │      0 │ train │     0 │
│ 4  │ train_acc  │ MulticlassAccuracy │      0 │ train │     0 │
│ 5  │ val_acc    │ MulticlassAccuracy │      0 │ train │     0 │
│ 6  │ test_acc   │ MulticlassAccuracy │      0 │ train │     0 │
│ 7  │ train_f1   │ MulticlassF1Score  │      0 │ train │     0 │
│ 8  │ val_f1     │ MulticlassF1Score  │      0 │ train │     0 │
│ 9  │ test_f1    │ MulticlassF1Score  │      0 │ train │     0 │
│ 10 │ convs      │ ModuleList         │ 98.7 K │ train │     0 │
│ 11 │ bns        │ ModuleList         │    768 │ train │     0 │
│ 12 │ act        │ GELU               │      0 │ train │     0 │
│ 13 │ drop_emb   │ Dropout            │      0 │ train │     0 │
│ 14 │ pool       │ AdaptiveMaxPool1d  │      0 │ train │     0 │
│ 15 │ head       │ Sequential         │  100 K │ train │     0 │
└────┴────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 852 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 852 K                                                                                                
Total estimated model params size (MB): 3                                                                          
Modules in train mode: 31                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=20` reached.


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.5853658318519592     │
│          test_f1          │    0.5853658318519592     │
│         test_loss         │    0.9241000413894653     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.9241000413894653,
  'test_acc': 0.5853658318519592,
  'test_f1': 0.5853658318519592}]